# Lectura de archivos – Indicador 5.1 (Pesca sostenible)

Este notebook lee automáticamente todos los archivos `.csv` (por año) y el archivo `.xlsx` que se encuentran en la carpeta `v5.1_fish_sostenible`.

- Los CSV se cargan en un diccionario `dfs_csv` (clave = año/filename).
- Además se construye un `df_csv_all` concatenado (con columna `source_file`).
- El XLSX se carga en `dfs_xlsx` (diccionario por hoja) y, si existe una hoja única, también en `df_xlsx`.

> Ajusta `BASE_DIR` si tu carpeta está en otra ubicación.


In [2]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# --- Configuración ---
BASE_DIR = Path.cwd()
BASE_DIR.exists(), BASE_DIR.resolve()


(True,
 PosixPath('/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/7NR-Indicators/indicadores/5.1_fish_sostenible'))

In [3]:
# Listar archivos esperados
csv_files = sorted(BASE_DIR.glob("*.csv"))
xlsx_files = sorted(BASE_DIR.glob("*.xlsx"))

csv_files, xlsx_files


([PosixPath('/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/7NR-Indicators/indicadores/5.1_fish_sostenible/2018.csv'),
  PosixPath('/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/7NR-Indicators/indicadores/5.1_fish_sostenible/2019.csv'),
  PosixPath('/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/7NR-Indicators/indicadores/5.1_fish_sostenible/2020.csv'),
  PosixPath('/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/7NR-Indicators/indicadores/5.1_fish_sostenible/2021.csv'),
  PosixPath('/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/7NR-Indicators/indicadores/5.1_fish_sostenible/2022.csv'),
  PosixPath('/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/7NR-Indicators/indicadores/5.1_fish_sostenible/2023.csv'),
  PosixPath('/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/7NR-Indicators/indicadores/5.1_fish_sostenible/2024.csv')],
 [PosixPath(

In [4]:
# Leer CSV por año (o por nombre de archivo)
dfs_csv = {}

for fp in csv_files:
    key = fp.stem  # ej: '2022'
    try:
        df = pd.read_csv(fp)
    except UnicodeDecodeError:
        # fallback común si viene en latin-1
        df = pd.read_csv(fp, encoding="latin-1")
    dfs_csv[key] = df

len(dfs_csv), list(dfs_csv.keys())[:10]


(7, ['2018', '2019', '2020', '2021', '2022', '2023', '2024'])

In [5]:
# Vista rápida de un CSV (elige el año/archivo)
example_key = next(iter(dfs_csv.keys()), 2020)
example_key


'2018'

In [6]:
if example_key is not None:
    display(dfs_csv[example_key].head())


,Especie,Estatus
0,Alfonsino,Agotada o colapsada
1,Anchoveta,Agotada o colapsada
2,Bacalao de profundidad,Agotada o colapsada
3,Besugo,Agotada o colapsada
4,Camarón Nailon,Plena explotación


In [7]:
# Concatenar todos los CSV (agrega columna source_file)
if dfs_csv:
    df_csv_all = (
        pd.concat(
            [df.assign(source_file=k) for k, df in dfs_csv.items()],
            ignore_index=True
        )
    )
else:
    df_csv_all = pd.DataFrame()

df_csv_all.shape


(140, 3)

In [8]:
df_csv_all.head()


,Especie,Estatus,source_file
0,Alfonsino,Agotada o colapsada,2018
1,Anchoveta,Agotada o colapsada,2018
2,Bacalao de profundidad,Agotada o colapsada,2018
3,Besugo,Agotada o colapsada,2018
4,Camarón Nailon,Plena explotación,2018


In [9]:
sorted(df_csv_all["Estatus"].unique())

['Agotada o colapsada',
 'Indeterminado',
 'Plena explotación',
 'Sobreexplotada',
 'Sobrexplotada',
 'Subexplotación',
 'Subexplotada']

In [10]:
# 1) Estandarizar categorías
map_estatus = {
    "Sobrexplotada": "Sobreexplotada",
    "Subexplotación": "Subexplotada",
}

df_csv_all["Estatus_std"] = df_csv_all["Estatus"].replace(map_estatus)

# chequeo rápido: deberían quedar 5-6 categorías (según si mantienes Indeterminado)
sorted(df_csv_all["Estatus_std"].unique())

['Agotada o colapsada',
 'Indeterminado',
 'Plena explotación',
 'Sobreexplotada',
 'Subexplotada']

In [11]:
# Conteo
counts = (
    df_csv_all
    .groupby(["source_file", "Estatus_std"])
    .size()
    .reset_index(name="n_especies")
)

# Total por año (usando transform)
counts["total_especies"] = counts.groupby("source_file")["n_especies"].transform("sum")
counts["proporcion"] = counts["n_especies"] / counts["total_especies"]

prop = counts[["source_file", "Estatus_std", "proporcion"]]
prop.head()

,source_file,Estatus_std,proporcion
0,2018,Agotada o colapsada,0.30
1,2018,Plena explotación,0.30
2,2018,Sobreexplotada,0.40
3,2019,Agotada o colapsada,0.20
4,2019,Indeterminado,0.05


In [13]:
tabla_prop = (
    prop
    .pivot(index="source_file", columns="Estatus_std", values="proporcion")
    .fillna(0)
    .sort_index()
)

print(tabla_prop.to_string())

Estatus_std  Agotada o colapsada  Indeterminado  Plena explotación  Sobreexplotada  Subexplotada
source_file                                                                                     
2018                        0.30           0.00               0.30            0.40          0.00
2019                        0.20           0.05               0.25            0.50          0.00
2020                        0.25           0.00               0.40            0.35          0.00
2021                        0.25           0.00               0.30            0.40          0.05
2022                        0.35           0.00               0.30            0.30          0.05
2023                        0.25           0.00               0.30            0.40          0.05
2024                        0.30           0.00               0.40            0.30          0.00
